In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from xgboost import XGBClassifier

In [ ]:
df_train=pd.read_csv("/content/train.csv")

In [ ]:
df_test=pd.read_csv("/content/test.csv")

In [ ]:
df_train.columns

Index(['ID_code', 'target', 'var_0', 'var_1', 'var_2', 'var_3', 'var_4',
       'var_5', 'var_6', 'var_7',
       ...
       'var_190', 'var_191', 'var_192', 'var_193', 'var_194', 'var_195',
       'var_196', 'var_197', 'var_198', 'var_199'],
      dtype='object', length=202)

In [ ]:
df_test.columns

Index(['ID_code', 'var_0', 'var_1', 'var_2', 'var_3', 'var_4', 'var_5',
       'var_6', 'var_7', 'var_8',
       ...
       'var_190', 'var_191', 'var_192', 'var_193', 'var_194', 'var_195',
       'var_196', 'var_197', 'var_198', 'var_199'],
      dtype='object', length=201)

In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Columns: 202 entries, ID_code to var_199
dtypes: float64(200), int64(1), object(1)
memory usage: 308.2+ MB


In [ ]:
df_train.drop(columns=["ID_code"],inplace=True)

In [ ]:
df_test.drop(columns=["ID_code"],inplace=True)

In [ ]:
def adding_rows_to_x(df):
    df_new=df.copy()
    df_new["row_mean"]=df_new.mean(axis=1)
    df_new["row_sd"]=df_new.std(axis=1)
    df_new["row_median"]=df_new.median(axis=1)
    df_new["row_min"]=df_new.min(axis=1)
    df_new["row_max"]=df_new.max(axis=1)
    df_new["row_sum"]=df_new.sum(axis=1)
    df_new["row_skew"]=df_new.skew(axis=1)
    df_new["pos_cnt"]=(df_new>0).sum(axis=1)
    df_new["neg_cnt"]=(df_new<0).sum(axis=1)
    return df_new

In [ ]:
x=df_train.drop(columns=["target"])
y=df_train["target"]

In [ ]:
X=adding_rows_to_x(x)

In [ ]:
X.shape

(200000, 209)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,random_state=365,stratify=y)

In [ ]:
df_train["target"].value_counts()

,count
target,
0,179902
1,20098


In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
ss=StandardScaler()

In [ ]:
xtrain_scaled=ss.fit_transform(xtrain)

In [ ]:
xtest_scale=ss.transform(xtest)

Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model1=LogisticRegression(solver='liblinear', random_state=365)

In [ ]:
params_grid = [
    {
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"],
        "C": [1.0, 0.1],
        "class_weight": ["balanced"]
    },
    {
        "solver": ["lbfgs"],
        "penalty": ["l2"],
        "C": [1.0, 0.1],
        "class_weight": ["balanced"]
    }
]

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
grid=GridSearchCV(
    estimator=model1,
    param_grid=params_grid,
    cv=3,
    scoring="accuracy"
)

In [ ]:
grid.fit(xtrain_scaled,ytrain)

GridSearchCV(cv=3,
             estimator=LogisticRegression(random_state=365, solver='liblinear'),
             param_grid=[{'C': [1.0, 0.1], 'class_weight': ['balanced'],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [1.0, 0.1], 'class_weight': ['balanced'],
                          'penalty': ['l2'], 'solver': ['lbfgs']}],
             scoring='accuracy')

In [ ]:
print(grid.best_params_)

{'C': 0.1, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}


In [ ]:
best_lr_model = grid.best_estimator_

# Make predictions on the scaled test data
y_pred_lr = best_lr_model.predict(xtest_scale)
y_pred_proba_lr = best_lr_model.predict_proba(xtest_scale)[:, 1]

# Evaluate the model
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(ytest, y_pred_lr))

print("\nClassification Report:")
print(classification_report(ytest, y_pred_lr))

print(f"\nAccuracy Score: {accuracy_score(ytest, y_pred_lr):.4f}")
print(f"ROC AUC Score: {roc_auc_score(ytest, y_pred_proba_lr):.4f}")

Confusion Matrix:
[[28347  7633]
 [  872  3148]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.79      0.87     35980
           1       0.29      0.78      0.43      4020

    accuracy                           0.79     40000
   macro avg       0.63      0.79      0.65     40000
weighted avg       0.90      0.79      0.82     40000


Accuracy Score: 0.7874
ROC AUC Score: 0.8666
